<a href="https://colab.research.google.com/github/M1NG0LL/Driver-Drowsiness-Detection-DDD/blob/model/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing

In [1]:
import os
import kagglehub
import random
import numpy as np
import hashlib
import tempfile
from PIL import Image
import shutil
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, Input, GlobalAveragePooling2D,
    BatchNormalization, Activation, Concatenate, Add, MaxPooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, f1_score
)
from collections import Counter
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# DataSet

installing dataset

In [4]:
data_dir = kagglehub.dataset_download('ismailnasri20/driver-drowsiness-dataset-ddd')

print('Dataset downloaded to:', data_dir)

Using Colab cache for faster access to the 'driver-drowsiness-dataset-ddd' dataset.
Dataset downloaded to: /kaggle/input/driver-drowsiness-dataset-ddd


In [5]:
!pip install datasets

In [6]:
from datasets import load_dataset
ds = load_dataset("n7i5x9/driver-drowsiness-dataset")
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/337M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/86.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18492 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2311 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2313 [00:00<?, ? examples/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x640 at 0x7D5329249760>, 'label': 0}


In [15]:
print(ds)

# Access the first training sample
first_train_sample = ds['train'][0]

# Print the keys available in the sample
print(f"\nKeys in the first training sample: {first_train_sample.keys()}")

# Display the image and its label
print(f"Label of the first training sample: {first_train_sample['label']}")
# display(first_train_sample['image'])

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 18492
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 2311
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2313
    })
})

Keys in the first training sample: dict_keys(['image', 'label'])
Label of the first training sample: 0


Split Data

In [18]:
import os
import shutil
import hashlib
import gc
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# =====================================================
# CONFIG
# =====================================================

# Kaggle dataset folder
kaggle_dir = "/kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)"

# Hugging Face DatasetDict
hf_ds = ds   # your printed DatasetDict

# Output merged folder
output_dir = "/kaggle/working/merged_dataset"

# Resize for consistency + lower storage
img_size = (300, 300)

# HF labels (edit if reversed)
label_map = {
    0: "Drowsy",
    1: "Non_Drowsy"
}

# Kaggle folder names mapping
folder_map = {
    "Drowsy": "Drowsy",
    "Non Drowsy": "Non_Drowsy",
    "Non_Drowsy": "Non_Drowsy",
    "Not_Drowsy": "Non_Drowsy"
}

# =====================================================
# START
# =====================================================

print("Starting merge")
print("Kaggle source :", kaggle_dir)
print("HF rows       :", sum(len(hf_ds[x]) for x in hf_ds.keys()))
print("Output folder :", output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "Drowsy"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "Non_Drowsy"), exist_ok=True)

# =====================================================
# HASH SYSTEM
# =====================================================

seen_hashes = set()

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def image_hash(img):
    return hashlib.md5(img.tobytes()).hexdigest()

# =====================================================
# 1. MERGE KAGGLE DATASET
# =====================================================

print("\nMerging Kaggle dataset")

kaggle_count = 0
kaggle_skip = 0

for folder in os.listdir(kaggle_dir):
    folder_path = os.path.join(kaggle_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    if folder not in folder_map:
        continue

    target_class = folder_map[folder]
    target_dir = os.path.join(output_dir, target_class)

    for file in tqdm(os.listdir(folder_path), desc=folder):
        src = os.path.join(folder_path, file)

        try:
            h = file_hash(src)

            if h in seen_hashes:
                kaggle_skip += 1
                continue

            seen_hashes.add(h)

            ext = Path(file).suffix.lower()
            new_name = f"kaggle_{kaggle_count}{ext}"

            shutil.copy2(src, os.path.join(target_dir, new_name))
            kaggle_count += 1

        except:
            kaggle_skip += 1

# =====================================================
# 2. MERGE HF DATASETDICT (RAM SAFE)
# =====================================================

print("\nMerging Hugging Face dataset")

hf_count = 0
hf_skip = 0

for split in hf_ds.keys():

    print("Processing split:", split)

    for i, item in enumerate(tqdm(hf_ds[split])):

        try:
            img = item["image"].convert("RGB")
            img = img.resize(img_size)

            label = label_map[item["label"]]
            save_dir = os.path.join(output_dir, label)

            h = image_hash(img)

            if h in seen_hashes:
                hf_skip += 1
                continue

            seen_hashes.add(h)

            save_path = os.path.join(save_dir, f"hf_{split}_{i}.jpg")
            img.save(save_path, quality=95)

            hf_count += 1

            if i % 500 == 0:
                gc.collect()

        except:
            hf_skip += 1

# =====================================================
# REPORT
# =====================================================

print("\nFinished Merge")
print("Kaggle copied :", kaggle_count)
print("HF copied     :", hf_count)
print("Skipped total :", kaggle_skip + hf_skip)

for cls in ["Drowsy", "Non_Drowsy"]:
    path = os.path.join(output_dir, cls)
    print(cls, ":", len(os.listdir(path)))

print("\nReady at:")
print(output_dir)

Starting merge
Kaggle source : /kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)
HF rows       : 23116
Output folder : /kaggle/working/merged_dataset

Merging Kaggle dataset


Drowsy: 100%|██████████| 22348/22348 [01:33<00:00, 238.01it/s]



Merging Hugging Face dataset
Processing split: train


100%|██████████| 18492/18492 [03:29<00:00, 88.34it/s] 


Processing split: validation


100%|██████████| 2311/2311 [00:20<00:00, 111.23it/s]


Processing split: test


100%|██████████| 2313/2313 [00:21<00:00, 105.78it/s]


Finished Merge
Kaggle copied : 41793
HF copied     : 19237
Skipped total : 3879
Drowsy : 33460
Non_Drowsy : 27570

Ready at:
/kaggle/working/merged_dataset


In [19]:
!pip install split-folders

In [21]:
import splitfolders

input_dir = '/kaggle/working/merged_dataset'
output_dir = "/dataset/working/splited_dataset"

splitfolders.ratio(
    input=input_dir,
    output=output_dir,
    seed=42,
    ratio=(.8, .15, .05)
)

Copying files: 61030 files [01:44, 583.84 files/s]


In [22]:
train_dir = os.path.join(output_dir, "train")
val_dir   = os.path.join(output_dir, "val")
test_dir  = os.path.join(output_dir, "test")

In [24]:
def get_all_files(base_dir):
    all_files = set()

    for class_name in os.listdir(base_dir):  # Drowsy / Not_Drowsy
        class_path = os.path.join(base_dir, class_name)

        if os.path.isdir(class_path):
            for file in os.listdir(class_path):
                full_path = os.path.join(class_name, file)
                all_files.add(full_path)

    return all_files


train_files = get_all_files(train_dir)
val_files = get_all_files(val_dir)
test_files = get_all_files(test_dir)


print("Train files Len:", len(train_files))
print("Val files Len:", len(val_files))
print("Test files Len:", len(test_files))
print("Train-Val Overlap:", len(train_files & val_files))
print("Train-Test Overlap:", len(train_files & test_files))
print("Val-Test Overlap:", len(val_files & test_files))

Train files Len: 48824
Val files Len: 9154
Test files Len: 3052
Train-Val Overlap: 0
Train-Test Overlap: 0
Val-Test Overlap: 0


# Data Augmentation

Variables

In [25]:
img_size = (300,300)
batch_size = 32

In [26]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,

    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.25,

    horizontal_flip=True,

    brightness_range=[0.7, 1.3],

    fill_mode='nearest'
)

test_val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [27]:
train_batches = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
val_batches = test_val_datagen.flow_from_directory(
    val_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
test_batches = test_val_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=False, seed=SEED
)

Found 48824 images belonging to 2 classes.
Found 9154 images belonging to 2 classes.
Found 3052 images belonging to 2 classes.


In [28]:
print("\n--- Class Distribution ---")
for name, batches in zip(['Train', 'Validation', 'Test'],
                         [train_batches, val_batches, test_batches]):
    counts = Counter(batches.classes)
    print(f"{name}: {dict(counts)}")


--- Class Distribution ---
Train: {np.int32(0): 26768, np.int32(1): 22056}
Validation: {np.int32(0): 5019, np.int32(1): 4135}
Test: {np.int32(0): 1673, np.int32(1): 1379}


# **Neural Network *(NN)***

Helper Methods

In [29]:
def conv_fn(x, filters, kernel, strides=1, padding='same'):
    """Conv2D + BatchNormalization + ReLU, returns the direct output."""
    x = Conv2D(filters, kernel, strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

Inception Customized Methods

In [30]:
def reduction_A_maker_fn(x):
    branch1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    branch2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(x)
    branch2 = BatchNormalization()(branch2)
    branch2 = Activation('relu')(branch2)

    branch3 = conv_fn(x, 256, 1, padding='same')
    branch3 = conv_fn(branch3, 256, 3, padding='same')
    branch3 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(branch3)
    branch3 = BatchNormalization()(branch3)
    branch3 = Activation('relu')(branch3)

    return Concatenate()([branch1, branch2, branch3])

def inception_resnet_B_maker_fn(x):
    in_channels = x.shape[-1]
    b1 = conv_fn(x, 192, 1)
    b2 = conv_fn(x, 128, 1)
    b2 = conv_fn(b2, 160, (1, 7))
    b2 = conv_fn(b2, 192, (7, 1))
    mixed = Concatenate()([b1, b2])
    mixed = Conv2D(in_channels, 1, padding='same', use_bias=True)(mixed)
    mixed = BatchNormalization()(mixed)
    return Activation('relu')(Add()([x, mixed]))

def reduction_B_maker_fn(x):
    b1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    b2 = conv_fn(x, 256, 1)
    b2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(b2)
    b2 = BatchNormalization()(b2)
    b2 = Activation('relu')(b2)

    b3 = conv_fn(x, 256, 1)
    b3 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b3)
    b3 = BatchNormalization()(b3)
    b3 = Activation('relu')(b3)

    b4 = conv_fn(x, 256, 1)
    b4 = conv_fn(b4, 256, 3, padding='same')
    b4 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b4)
    b4 = BatchNormalization()(b4)
    b4 = Activation('relu')(b4)

    return Concatenate()([b1, b2, b3, b4])


Build Hybrid Model

In [31]:
def build_model(input_shape=(300, 300, 3)):
    inputs = Input(shape=input_shape)
    backbone = EfficientNetB3(include_top=False, weights='imagenet', input_tensor=inputs)
    x = backbone.output

    # Add custom blocks
    x = reduction_A_maker_fn(x)
    x = inception_resnet_B_maker_fn(x)

    x = reduction_B_maker_fn(x)

    # Classification head
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model, backbone

In [32]:
model, backbone = build_model()
backbone.trainable = False

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **Train**

In [33]:
class_counts = Counter(train_batches.classes)
total = sum(class_counts.values())
class_weight = {
    0: total / (2 * class_counts[0]),
    1: total / (2 * class_counts[1])
}
print(f"\nClass weights: {class_weight}")


Class weights: {0: 0.9119844590555888, 1: 1.1068190061661225}


In [34]:
model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_auc', patience=6, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=4, mode='max', min_lr=1e-6),
    ModelCheckpoint('best_model_phase1.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

Phase 1: Training the Head

In [35]:
history1 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks_phase1
)

Epoch 1/15
1209/1526 ━━━━━━━━━━━━━━━━━━━━ 4:51 920ms/step - accuracy: 0.7833 - auc: 0.8633 - loss: 0.4970 - precision: 0.7653 - recall: 0.7562

KeyboardInterrupt: 

Phase 2: Fine-Tuning

In [ ]:
model = keras.models.load_model('best_model_phase1.h5', compile=False)

for layer in model.layers:
    layer.trainable = False

for layer in model.layers:
    if layer.name.startswith("block6") or layer.name.startswith("top") or layer.name.startswith("dense"):
        layer.trainable = True
        print(f"Layer {layer.name} is trainable")

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=5, mode='max', min_lr=1e-7),
    ModelCheckpoint('best_model_finetuned.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

In [ ]:
history2 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=12,
    class_weight=class_weight,
    callbacks=callbacks_phase2
)

In [ ]:
model = keras.models.load_model('best_model_finetuned.h5')

# Plot Training Curves

In [ ]:
def plot_training(history, title):
    metrics = ['loss', 'accuracy', 'precision', 'recall', 'auc']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, metric in enumerate(metrics):
        ax = axes[i]
        ax.plot(history.history[metric], label='Train')
        ax.plot(history.history[f'val_{metric}'], label='Val')
        ax.set_title(f'{metric.upper()} - {title}')
        ax.legend()
    # Hide the sixth (empty) subplot
    axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

plot_training(history1, 'Phase 1')
plot_training(history2, 'Phase 2 (Fine-Tuning)')

# Evaluate

In [ ]:
print("\n--- Evaluation on Test Set ---")
test_loss, test_acc, test_prec, test_rec, test_auc = model.evaluate(test_batches, verbose=0)
print(f"Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, AUC: {test_auc:.4f}")

y_pred_probs = model.predict(test_batches).flatten()
y_true = test_batches.classes


# Threshold Tuning

In [ ]:
val_probs = model.predict(val_batches).flatten()
val_true = val_batches.classes

precisions, recalls, thresholds = precision_recall_curve(val_true, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold based on F1-score (validation set): {best_thresh:.4f}")

y_pred_opt = (y_pred_probs > best_thresh).astype(int)

print("\n--- Classification Report (Default Threshold 0.5) ---")
print(classification_report(y_true, (y_pred_probs > 0.5).astype(int),
                            target_names=['Non Drowsy', 'Drowsy']))

print("\n--- Classification Report (Optimized Threshold) ---")
print(classification_report(y_true, y_pred_opt,
                            target_names=['Non Drowsy', 'Drowsy']))

# Plot confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred_opt), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non Drowsy', 'Drowsy'],
            yticklabels=['Non Drowsy', 'Drowsy'])
plt.title('Confusion Matrix (Optimized Threshold)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Test-Time Augmentation (TTA)

In [ ]:
def predict_with_tta(model, directory, target_size, batch_size, n_aug=5):
    """
    Predict using an average of n_aug random augmentations on the test data.
    Note: The directory must contain the same subfolder structure as the training data.
    """
    tta_datagen = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        rotation_range=10,
        zoom_range=0.1,
        brightness_range=[0.9, 1.1]
    )
    all_probs = []
    for i in range(n_aug):
        print(f"TTA round {i+1}/{n_aug}")
        gen = tta_datagen.flow_from_directory(
            directory,
            target_size=target_size,
            batch_size=batch_size,
            class_mode=None,      # we don't need labels
            shuffle=False
        )
        probs = model.predict(gen)
        all_probs.append(probs.flatten())
    return np.mean(all_probs, axis=0)


In [ ]:
print("\n--- Applying Test-Time Augmentation ---")
y_pred_tta_probs = predict_with_tta(model, test_dir, img_size, batch_size, n_aug=5)
y_pred_tta = (y_pred_tta_probs > best_thresh).astype(int)
print(classification_report(y_true, y_pred_tta))